# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and describes protein abundance and modifications identified using mass spectrometry analysis of human extracellular vesicles.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Access and display main metadata fields
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}\nPublished: {metadata.datePublished}")
print(f"License: {metadata.license}\nIdentifier: {metadata.identifier}")
print("Cite as:", metadata.citeAs)

## 2. Data Overview
Review available record sets, their fields, and columns, referenced by their `@id` values.

In [ ]:
# List all record sets and their fields by their @id
print("Available record sets in the dataset:")
record_sets = []
for record_set in dataset.record_sets:
    print(f"- Record set name: {getattr(record_set, 'name', None)}")
    print(f"  @id: {record_set.id}")
    record_sets.append(record_set.id)
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {getattr(field, 'name', None)} (field @id: {field.id})")
    print("  Columns:")
    for column in record_set.columns:
        print(f"    - {getattr(column, 'name', None)} (column @id: {column.id})")
    print()
# Choose main record set for demonstration
primary_record_set_id = record_sets[0] if record_sets else None

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis, using record set and field `@id`s discovered above.

In [ ]:
# Extract records from all main record sets
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records for record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    print(f"- {df.shape[0]} rows, {df.shape[1]} columns loaded.")
    print(f"- Columns: {df.columns.tolist()}")
    dataframes[record_set_id] = df
    print()
# If there are many record sets, choose the largest/main for demonstration
main_rs = primary_record_set_id
print(f"Showing the first five records from main record set @id: {main_rs}")
dataframes[main_rs].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering numeric fields, normalizing, and grouping. All field and column references use their `@id`.

In [ ]:
# Identify a numeric field by @id
# To help, print all columns and pick a likely numeric one ('cr:molecular_weight' is common, otherwise use another numeric @id)
df = dataframes[main_rs]
print(f"Columns in DataFrame for @id={main_rs}:")
for c in df.columns:
    print(c)

# You may need to adjust this field name if dataset changes
numeric_field_id = None
for col in df.columns:
    if 'weight' in col or 'MW' in col or 'abundance' in col or 'count' in col or 'coverage' in col:
        numeric_field_id = col
        break
if numeric_field_id is None:
    numeric_field_id = df.columns[0]  # fallback: just use first field

print(f"Using field '@id': {numeric_field_id} as numeric field.")

# Try converting to numeric
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filter records where value exceeds a threshold
threshold = df[numeric_field_id].mean()
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records in '{numeric_field_id}' > {threshold:.2f} (mean): {len(filtered_df)} rows.")

# Normalize the selected numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Identify a group field (try by name or select first string column)
group_field_id = None
for c in df.columns:
    if 'protein' in c.lower() or 'accession' in c.lower() or 'description' in c.lower() or 'sample' in c.lower():
        group_field_id = c
        break
if group_field_id is None:
    group_field_id = df.columns[1] if len(df.columns) > 1 else df.columns[0]

print(f"Grouping by field '@id': {group_field_id}")
# Group and aggregate
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print("Grouped mean values:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions and relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If grouping field is reasonably sized, show means per group
if 'grouped_df' in locals() and grouped_df.shape[0] < 30:
    plt.figure(figsize=(10,5))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=90)
    plt.show()

## 6. Conclusion
In this notebook, we used the FAIR² protein abundance and modification dataset and demonstrated:
- Loading Croissant metadata and record sets using their `@id`
- Viewing and extracting fields and columns with structural data overview
- Performing basic EDA and visualizations to identify patterns in protein mass or abundance

This workflow can be adapted to other Croissant-structured datasets using the `mlcroissant` package and referencing data entities by their unique `@id` identifiers.